In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l8.6 Speculative decoding

A small **draft** model proposes several tokens cheaply; the big **target** model
verifies them in a single pass and accepts the longest correct prefix. When draft
and target agree often, you get the target's quality at a fraction of its steps,
with no change to the output distribution.

In [ ]:
from pocketlm import train_tiny, pick_config

corpus = open(CORPUS_PATH, encoding="utf-8").read()
target, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
draft, _ = train_tiny(corpus, pick_config(DEVICE, 1), seed=1)   # a different, weaker model
ctx = torch.tensor([tok.encode("The ")])
proposed = []
d = ctx.clone()
for _ in range(4):                       # draft proposes 4 tokens greedily
    dl, _ = draft(d[:, -draft.cfg.block_size:])
    nt = dl[:, -1].argmax(-1, keepdim=True)
    d = torch.cat([d, nt], dim=1)
    proposed.append(int(nt.item()))
accepted = 0
t = ctx.clone()
for tok_id in proposed:                  # target verifies greedily
    tl, _ = target(t[:, -target.cfg.block_size:])
    if int(tl[:, -1].argmax(-1).item()) == tok_id:
        accepted += 1
        t = torch.cat([t, torch.tensor([[tok_id]])], dim=1)
    else:
        break
print(f"draft proposed {len(proposed)}, target accepted {accepted}")

Every accepted token skipped a full target step. The target still has the final
say on each token, so correctness is preserved, the speedup is pure when agreement
is high.

In [ ]:
assert 0 <= accepted <= len(proposed)